©2026. For information, contact Deloitte Tohmatsu Group.

# 📝 演習概要

この演習では、PyTorchを用いてBERTモデルを実装し、IMDb映画レビューの感情分類（ポジティブ/ネガティブ）を行います。Tokenizer、Embedding、LayerNorm、Multi-Head Attentionなど各モジュールを構築し、事前学習済みモデルのファインチューニングまでの流れを学びます。

# 事前準備

[JDLAが策定しているバージョン](https://www.jdla.org/certificate/engineer/)に合わせるために、以下のセルの実行をお願いします．

（#コメントアウト されているものは必要ありません）

また実行完了後に「ランタイムの再起動」をして下さい．

（以下のセルの実行は、最初にしていただければ、以降必要ありません．）

In [ ]:
%%capture
# !pip uninstall matplotlib -y
# !pip install matplotlib==3.9.4

# !pip uninstall opencv-python -y
# !pip install opencv-python==4.11.0.86

!pip uninstall torch -y
!pip install torch==2.7.0 -q

# !pip uninstall torchvision -y
# !pip install torchvision==0.22.0 -q

今回使用する「torhtext」ライブラリが正常に実行するように以下のセルを実行してください．<br />
※torchvision の 互換性でエラーが出る場合がありますが、利用していないため、そのままご利用ください.

In [ ]:
# # 既存の関連ライブラリをアンインストール
# !pip uninstall torchtext -y
# !pip uninstall torchvision torchaudio -y

# # 互換性のあるバージョンで再インストール
# !pip install torchtext==0.15.2 -q

# !pip install torchdata==0.6.1 -q
# !pip install AttrDict3 -q
# !pip install transformers==4.31.0 -q

!pip uninstall -y torchtext torchvision torchaudio transformers tokenizers

# torchvision/torchaudio は torch==2.7.0 に揃える
!pip install -q "torchvision==0.22.0" "torchaudio==2.7.0"

# torch==2.7.0 と噛み合う torchdata（0.6.1は2.0系用）
!pip install -q "torchdata==0.11.0"

# transformers は Py3.12で安定する版に上げ、tokenizers のホイールも拾う
!pip install -q "transformers==4.41.0" "tokenizers>=0.15.2"

# ついでに指定パッケージ
!pip install -q AttrDict3

Found existing installation: torchvision 0.23.0+cu126
Uninstalling torchvision-0.23.0+cu126:
  Successfully uninstalled torchvision-0.23.0+cu126
Found existing installation: torchaudio 2.8.0+cu126
Uninstalling torchaudio-2.8.0+cu126:
  Successfully uninstalled torchaudio-2.8.0+cu126
Found existing installation: transformers 4.57.1
Uninstalling transformers-4.57.1:
  Successfully uninstalled transformers-4.57.1
Found existing installation: tokenizers 0.22.1
Uninstalling tokenizers-0.22.1:
  Successfully uninstalled tokenizers-0.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 61.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 116.2 MB/s eta 0:00:00


# BRRT　コード演習

BERTと呼ばれるディープラーニングモデルを使用し、IMDbデータセットに対してその内容がポジティブなのかネガティブなのか2値のクラス分類を行う感情分析に取り組みます。

---

目次
1.   事前準備
   * 準備ファイル
   * IMDbをダウンロード
   * fastTextの英語学習済みモデルをダウンロード
   * IMDb(Internet Movie Database)のDataLoderを実装
   * Tokenizerの実装
2.   BERTの実装
   * BERT_BASEのネットワークの設定ファイルの読み込み
   * BERT用にLayerNormalization層を定義
   * Embeddingsモジュールの実装
   * BertLayerモジュール
   * BertLayerモジュールの繰り返し部分
   * BertPoolerモジュール
   * 動作確認
   * BERTモデルの作成
3.   BERTを用いたベクトル表現の比較
   * 学習済みモデルのロード
   * BERT用のTokenizerの実装
   * Bankの文脈による意味変化を単語ベクトルとして求める
4.   BERTの学習・推論、判定根拠の可視化を実装
   * IMDbデータを読み込み、DataLoaderを作成（BERTのTokenizerを使用）
   * 感情分析用のBERTモデルを構築
   * BERTのファインチューニング向けた設定
   * 学習・検証を実施
   * Attentionの可視化



# 1. 事前準備


### 準備ファイル

In [ ]:
import os
import glob
import io
import json
import math
import string
import urllib.request
import zipfile
import tarfile
import unicodedata
import random
import re
import collections
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import (
    BertConfig,
    BertTokenizer,
    BertForSequenceClassification
)

from tqdm import tqdm
from sklearn.model_selection import train_test_split
from IPython.display import HTML

# 辞書変数をオブジェクト変数にする
from attrdict import AttrDict


In [ ]:
# 乱数のシードを設定（再現性のため）
torch.manual_seed(1234)
np.random.seed(1234)
random.seed(1234)

In [ ]:
# フォルダ「data」が存在しない場合に作成する
data_dir = "./data/"
if not os.path.exists(data_dir):
    os.mkdir(data_dir)

In [ ]:
# フォルダ「vocab」が存在しない場合は作成する
vocab_dir = "./vocab/"
if not os.path.exists(vocab_dir):
    os.mkdir(vocab_dir)

In [ ]:
# フォルダ「weights」が存在しない場合は作成する
weights_dir = "./weights/"
if not os.path.exists(weights_dir):
    os.mkdir(weights_dir)

In [ ]:
# 単語集：ボキャブラリーをダウンロード

# 'bert-base-uncased':
# https://s3.amazonaws.com/models.huggingface.co/bert/bert-base-uncased-vocab.txt

save_path="./vocab/bert-base-uncased-vocab.txt"
url = "https://s3.amazonaws.com/models.huggingface.co/bert/bert-base-uncased-vocab.txt"
urllib.request.urlretrieve(url, save_path)

('./vocab/bert-base-uncased-vocab.txt',
 <http.client.HTTPMessage at 0x7c7251d5ef00>)

In [ ]:
# BERTの学習済みモデル 'bert-base-uncased'
# https://github.com/huggingface/pytorch-pretrained-BERT/
# https://s3.amazonaws.com/models.huggingface.co/bert/bert-base-uncased.tar.gz

# ダウンロード
save_path = "./weights/bert-base-uncased.tar.gz"
url = "https://s3.amazonaws.com/models.huggingface.co/bert/bert-base-uncased.tar.gz"
urllib.request.urlretrieve(url, save_path)

# 解凍
archive_file = "./weights/bert-base-uncased.tar.gz"  # Uncasedは小文字化モードという意味です
tar = tarfile.open(archive_file, 'r:gz')
tar.extractall('./weights/')  # 解凍
tar.close()  # ファイルをクローズ

# フォルダ「weights」に「pytorch_model.bin」と「bert_config.json」ができます

/tmp/ipython-input-2031734220.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('./weights/')  # 解凍


### IMDb(Internet Movie Database:映画のレビュー文章を集めたデータセット)をダウンロード

http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
    

In [ ]:
# IMDbをダウンロード

url = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
save_path = "./data/aclImdb_v1.tar.gz"
if not os.path.exists(save_path):
    urllib.request.urlretrieve(url, save_path)

In [ ]:
# './data/aclImdb_v1.tar.gz'の解凍

# tarファイルを読み込み
tar = tarfile.open('./data/aclImdb_v1.tar.gz')
tar.extractall('./data/')  # 解凍
tar.close()  # ファイルをクローズ

# フォルダ「data」内にフォルダ「aclImdb」というものができます。

/tmp/ipython-input-502538240.py:5: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('./data/')  # 解凍


## fastTextの英語学習済みモデルをダウンロード

ボキャブラリー作成のために必要となる。

In [ ]:
# fastTextの公式の英語学習済みモデル（650MB）をダウンロード
url = "https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip"
save_path = "./data/wiki-news-300d-1M.vec.zip"
if not os.path.exists(save_path):
    urllib.request.urlretrieve(url, save_path)

In [ ]:
# フォルダ「data」内の「/wiki-news-300d-1M.vec.zip」を解凍する

Zip = zipfile.ZipFile("./data/wiki-news-300d-1M.vec.zip")
Zip.extractall("./data/")  # ZIPを解凍
Zip.close()  # ZIPファイルをクローズ

# フォルダ「data」内にフォルダ「wiki-news-300d-1M.vec」ができているかを確認する


 ## IMDb(Internet Movie Database)のDataLoderを実装

**IMDbデータセットをtsv形式に変換する**

tsv形式とは、１行がひとつのdataを示し、そのなかでテキストとラベルが記載され、それらをタブで区切ったファイル。

In [ ]:
#訓練データのtsvファイルを作成する。
f = open('./data/IMDb_train.tsv', 'w')

#ポジティブデータ
path = './data/aclImdb/train/pos/'
for fname in glob.glob(os.path.join(path, '*.txt')):
  with io.open(fname, 'r', encoding='utf-8') as ff:
    text = ff.readline()

    # タブがあれば消す
    text = text.replace('\t', ' ')

    #テキストとラベルを記載し、タグで区切ったファイルにする
    text = text +'\t'+'1'+'\t'+'\n'
    f.write(text)

#ネガティブデータ
path = '.data/aclImdb/train/neg/'
for fname in glob.glob(os.path.join(path, '*.txt')):
  with io.open(fname, 'r', encoding='utf-8') as ff:
    text = ff.readline()

    # タブがあれば消す
    text = text.replace('\t', ' ')

    #テキストとラベルを記載し、タグで区切ったファイルにする。
    text = text+'\t'+'0'+'\t'+'\n'
    f.write(text)

f.close()

In [ ]:
# テストデータの作成

f = open( './data/IMDb_test.tsv', 'w')

#ポジティブデータ
path = './data/aclImdb/test/pos/'
for fname in glob.glob(os.path.join(path, '*.txt')):
   with io.open(fname, 'r', encoding='utf-8') as ff:
     text = ff.readline()

     #タブがあれば消す
     text = text.replace('\t', ' ')

     #テキストとラベルを記載し、タグで区切ったファイルにする
     text = text+'\t'+'1'+'\t'+'\n'
     f.write(text)

#ネガティブデータ
path = './data/aclIMdb/test/neg/'
for fname in glob.glob(os.path.join(path, '*.txt')):
  with io.open(fname, 'r', encoding='utf-8') as ff:
    text = ff.readline()

    #タブがあれば消す
    text = text.replace('\t', ' ')

    #テキストとラベルを記載し、タグで区切ったファイルにする。
    text = text+'\t'+'0'+'\t'+'\n'
    f.write(text)

f.close()


##Tokenizerの実装

In [ ]:
class BasicTokenizer(object):
    """
    基本的なトークン化を実装する (句読点による分割、小文字化など)
    """

    def __init__(self,
                 do_lower_case=True,
                 never_split=("[UNK]", "[SEP]", "[PAD]", "[CLS]", "[MASK]")):
        """BasicTokenizerを構築する。
        引数は以下の通り：
          do_lower_case ：入力を小文字にするかどうか。
        """
        self.do_lower_case = do_lower_case
        self.never_split = never_split

    def tokenize(self, text):
        """テクストの一部をトークン化する。"""
        text = self._clean_text(text)               # 制御文字の除去・空白の正規化
        text = self._tokenize_chinese_chars(text)   # CJK文字の前後にスペースを足す
        orig_tokens = whitespace_tokenize(text)     # 空白で一旦分割
        split_tokens = []
        for token in orig_tokens:
            # do_lower_case=True かつ 特殊トークンでなければ
            # - 小文字化
            # - アクセント除去
            if self.do_lower_case and token not in self.never_split:
                token = token.lower()
                token = self._run_strip_accents(token)
            # 句読点でさらに分割
            split_tokens.extend(self._run_split_on_punc(token))

        # 句読点分割した結果をもう一度空白区切り→最終トークン列にする
        output_tokens = whitespace_tokenize(" ".join(split_tokens))
        return output_tokens

    def _run_strip_accents(self, text):
        """テキストの一部からアクセントを取り除く。"""
        # 'é' → 'e' のようにダイアクリティカルマークを削る処理
        # unicodedata.normalize("NFD") で分解してMnカテゴリ(結合文字)を落とす
        text = unicodedata.normalize("NFD", text)
        output = []
        for char in text:
            cat = unicodedata.category(char)
            if cat == "Mn":     # Mn = Nonspacing Mark (結合分音記号など)
                continue
            output.append(char)
        return "".join(output)

    def _run_split_on_punc(self, text):
        """句読点を分割する"""
        if text in self.never_split:
            return [text]
        chars = list(text)
        i = 0
        start_new_word = True
        output = []
        while i < len(chars):
            char = chars[i]
            if _is_punctuation(char):   # 句読点と判定されたら、それだけを独立トークンに
                output.append([char])
                start_new_word = True
            else:
                if start_new_word:
                    output.append([])   # 新しいトークン開始
                start_new_word = False
                output[-1].append(char)
            i += 1

        return ["".join(x) for x in output]

    def _tokenize_chinese_chars(self, text):
        """CJK文字列に空白を追加する"""
        # CJK統合漢字などの前後にスペースを挟んで、他のアルファベットとくっつかないようにする
        # → 後段での whitespace_tokenize() が素直に働くようにする
        output = []
        for char in text:
            cp = ord(char)
            if self._is_chinese_char(cp):
                output.append(" ")
                output.append(char)
                output.append(" ")
            else:
                output.append(char)
        return "".join(output)

    def _is_chinese_char(self, cp):
        """CP が日中韓文字のコードポイントであるかどうかをチェックする"""
        # これは「漢字」を日中韓ユニコードブロック内のものとして定義している。
        # https://en.wikipedia.org/wiki/CJK_Unified_Ideographs_(Unicode_block)
        # 日中韓ユニコードブロックは日本語と韓国語のすべての文字ではないことに注意が必要。
        # 日本語のひらがなとカタカナが異なるブロックにあるように、現代のハングルも別ブロックである。
        # なお、これらのアルファベットは スペースで区切られた単語なので、特別扱いされずに処理される。
        if ((cp >= 0x4E00 and cp <= 0x9FFF) or  #
                (cp >= 0x3400 and cp <= 0x4DBF) or  #
                (cp >= 0x20000 and cp <= 0x2A6DF) or  #
                (cp >= 0x2A700 and cp <= 0x2B73F) or  #
                (cp >= 0x2B740 and cp <= 0x2B81F) or  #
                (cp >= 0x2B820 and cp <= 0x2CEAF) or
                (cp >= 0xF900 and cp <= 0xFAFF) or  #
                (cp >= 0x2F800 and cp <= 0x2FA1F)):  #
            return True

        return False

    def _clean_text(self, text):
        """テキスト上で無効な文字の削除と空白のクリーンアップを実行する"""
        # 制御文字などモデルに入れたくない文字を落とし、
        # タブ/改行などの空白類は" "に正規化する
        output = []
        for char in text:
            cp = ord(char)
            if cp == 0 or cp == 0xfffd or _is_control(char):
                continue
            if _is_whitespace(char):
                output.append(" ")
            else:
                output.append(char)
        return "".join(output)



class WordpieceTokenizer(object):
    """文字のトークン化"""
    # WordPiece分割を行うトークナイザ
    # - 事前にBasicTokenizerなどで空白＆句読点レベルまで分割されたトークンをさらに細かく砕く
    # - 未知語をサブワード列にする ("unaffable" → "un", "##aff", "##able")
    # - 語彙にない文字列は [UNK] にフォールバックする
    # BERTのサブワード化そのもの

    def __init__(self, vocab, unk_token="[UNK]", max_input_chars_per_word=100):
        self.vocab = vocab                              # サブワード辞書 (token -> id のもとになる)
        self.unk_token = unk_token                      # 分割できなかったときに使うトークン
        self.max_input_chars_per_word = max_input_chars_per_word  # 1単語が長すぎる場合の上限

    def tokenize(self, text):
        """テキストの一部を単語の断片にトークン化する。
        これは、トークン化を実行するために、最長一致優先貪欲アルゴリズムを使用する。
        与えられた語彙を使って
        例えば
          input = "unaffable"
          output = ["un", "##aff", "##able"]
　　　　のように与えられる。
　　　　引数は以下になる。
          text：単一のトークンまたは空白で区切られたトークン。既に `BasicTokenizer` を通過している必要がある。
        戻り値は単語トークンのリストになる。
        """

        output_tokens = []
        for token in whitespace_tokenize(text):  # 空白で区切られたトークン単位で処理
            chars = list(token)
            # 単語があまりにも長い場合(例:100文字超など)は即[UNK]扱い
            if len(chars) > self.max_input_chars_per_word:
                output_tokens.append(self.unk_token)
                continue

            is_bad = False      # 分割に失敗したかどうか
            start = 0
            sub_tokens = []
           # Greedyに左から右へ進む
            while start < len(chars):
                end = len(chars)
                cur_substr = None
                while start < end:
                    substr = "".join(chars[start:end])
                    if start > 0:
                        substr = "##" + substr  # 先頭以外は "##" を頭につける(BERT形式)
                    if substr in self.vocab:    # 辞書にあればそれを採用
                        cur_substr = substr
                        break
                    end -= 1
                if cur_substr is None:
                  # どこまで削ってもヒットしなかった→このトークン全体を[UNK]扱い
                    is_bad = True
                    break
                sub_tokens.append(cur_substr)
                start = end  # マッチした位置の次から再開

            if is_bad:
                output_tokens.append(self.unk_token)  # 未知語扱い
            else:
                output_tokens.extend(sub_tokens)      # 見つけたサブワード列を追加
        return output_tokens



def _is_whitespace(char):
    """文字が空白文字であるかどうかをチェックする。"""
    # \t, \n, and \r are technically contorl characters but we treat them
    # as whitespace since they are generally considered as such.
    # タブ・改行・復帰(\r)も「空白」とみなす
    if char == " " or char == "\t" or char == "\n" or char == "\r":
        return True
    cat = unicodedata.category(char)  # Unicodeカテゴリを取得
    if cat == "Zs":
        return True
    return False


def _is_control(char):
    """Checks whether `chars` is a control character."""
    # これらは技術的には制御文字だが、 空白文字としてカウントする。
    # 制御文字(不可視)かどうかを判定するが、タブ/改行/復帰は例外でFalseにする
    if char == "\t" or char == "\n" or char == "\r":
        return False
    cat = unicodedata.category(char)
    if cat.startswith("C"):  # Unicodeカテゴリが "C*" なら制御文字系
        return True
    return False


def _is_punctuation(char):
    """文字が句読点かどうか判別する"""
    cp = ord(char)
    # 文字や数字以外のASCII文字は全て句読点として扱われる。
    # "^", "$", "`"などの文字はUnicodeの句読点クラスには含まれていないが, 一貫性を保つためにいずれにしても句読点として扱う。
    if ((cp >= 33 and cp <= 47) or (cp >= 58 and cp <= 64) or
            (cp >= 91 and cp <= 96) or (cp >= 123 and cp <= 126)):
        return True
    cat = unicodedata.category(char)
    if cat.startswith("P"):
        return True
    return False


def whitespace_tokenize(text):
    """基本的な空白文字と句読点をトークン化する"""
    text = text.strip()
    if not text:
        return []
    tokens = text.split()
    return tokens

#2. BERTの実装

## BERT_Baseのネットワークの設定ファイルの読み込み

Transformerが12段であることや、特徴量ベクトルが768次元であることなどを記載したネットワーク設定ファイル「bert_config.json」を読み込む。

なお、jsonファイルを扱いやすくするために、パッケージattrdictを利用して、辞書型変数ではなく、クラスオブジェクトに変換する。このことで、config['hidden_size']と記載していたものを、config.hidden_sizeで設定パラメータにアクセスできるようになる。

In [ ]:
config_file = './weights/bert_config.json'

# ファイルを開き、JSONとして読み込み
json_file = open(config_file, 'r')
config = json.load(json_file)

# 出力関数
config

{'attention_probs_dropout_prob': 0.1,
 'hidden_act': 'gelu',
 'hidden_dropout_prob': 0.1,
 'hidden_size': 768,
 'initializer_range': 0.02,
 'intermediate_size': 3072,
 'max_position_embeddings': 512,
 'num_attention_heads': 12,
 'num_hidden_layers': 12,
 'type_vocab_size': 2,
 'vocab_size': 30522}

In [ ]:
config = AttrDict(config)
config.hidden_size

768

## BERT用にLayerNormalization層を定義

In [ ]:
#BERT用にLayerNormalization層を定義する
#実装の細かな点をTensorFlowにあわせていく必要がある
class BertLayerNorm(nn.Module):
  'LayerNormalization層'

  def __init__(self, hidden_size, eps=1e-12):
    super(BertLayerNorm, self).__init__()
    self.gamma = nn.Parameter(torch.ones(hidden_size))#weightのこと
    self.beta = nn.Parameter(torch.zeros(hidden_size))#biasのこと
    self.variance_epsilon = eps

  def forward(self, x):
    u = x.mean(-1, keepdim=True)
    s = (x - u).pow(2).mean(-1, keepdim=True)
    x = (x - u) / torch.sqrt(s + self.variance_epsilon)
    return self.gamma * x + self.beta

## Embeddingsモジュールの実装

In [ ]:
# BERTのEmbeddingsモジュール
class BertEmbeddings(nn.Module):
    """文章の単語ID列と、1文目か2文目かの情報を、埋め込みベクトルに変換する
    """

    def __init__(self, config):
        super(BertEmbeddings, self).__init__()

        # 3つのベクトル表現の埋め込み

        # Token Embedding：単語IDを単語ベクトルに変換
        # vocab_size = 30522：BERTの学習済みモデルで使用したボキャブラリーの量
        # hidden_size = 768 ：特徴量ベクトルの長さは768
        self.word_embeddings = nn.Embedding(
            config.vocab_size, config.hidden_size, padding_idx=0)
        # （注釈）padding_idx=0はidx=0の単語のベクトルは0にする。BERTのボキャブラリーのidx=0が[PAD]である。

        # Transformer Positional Embedding：位置情報テンソルをベクトルに変換
        # Transformerの場合はsin、cosからなる固定値だったが、BERTは学習させる
        # max_position_embeddings = 512　で文の長さは512単語
        self.position_embeddings = nn.Embedding(
            config.max_position_embeddings, config.hidden_size)

        # Sentence Embedding：文章の1文目、2文目の情報をベクトルに変換
        # type_vocab_size = 2
        self.token_type_embeddings = nn.Embedding(
            config.type_vocab_size, config.hidden_size)

        # 作成したLayerNormalization層
        self.LayerNorm = BertLayerNorm(config.hidden_size, eps=1e-12)

        # Dropout 'hidden_dropout_prob': 0.1
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, input_ids, token_type_ids=None):
        '''
        input_ids： [batch_size, seq_len]の文章の単語IDの羅列
        token_type_ids：[batch_size, seq_len]の各単語が1文目なのか、2文目なのかを示すid
        '''

        # 1. Token Embeddings
        # 単語IDを単語ベクトルに変換
        words_embeddings = self.word_embeddings(input_ids)

        # 2. Sentence Embedding
        # token_type_idsがない場合は文章の全単語を1文目として、0にする
        # そこで、input_idsと同じサイズのゼロテンソルを作成
        if token_type_ids is None:
            token_type_ids = torch.zeros_like(input_ids)
        token_type_embeddings = self.token_type_embeddings(token_type_ids)

        # 3. Transformer Positional Embedding：
        # [0, 1, 2 ・・・]と文章の長さだけ、数字が1つずつ昇順に入った
        # [batch_size, seq_len]のテンソルposition_idsを作成
        # position_idsを入力して、position_embeddings層から768次元のテンソルを取り出す
        seq_length = input_ids.size(1)  # 文章の長さ
        position_ids = torch.arange(
            seq_length, dtype=torch.long, device=input_ids.device)
        position_ids = position_ids.unsqueeze(0).expand_as(input_ids)
        position_embeddings = self.position_embeddings(position_ids)

        # 3つの埋め込みテンソルを足し合わせる [batch_size, seq_len, hidden_size]
        embeddings = words_embeddings + position_embeddings + token_type_embeddings

        # LayerNormalizationとDropoutを実行
        embeddings = self.LayerNorm(embeddings)
        embeddings = self.dropout(embeddings)

        return embeddings

## BERTレイヤーモジュール

In [ ]:
class BertLayer(nn.Module):
    '''BERTのBertLayerモジュールです。Transformerになります'''

    def __init__(self, config):
        super(BertLayer, self).__init__()

        # Self-Attention部分
        self.attention = BertAttention(config)

        # Self-Attentionの出力を処理する全結合層
        self.intermediate = BertIntermediate(config)

        # Self-Attentionによる特徴量とBertLayerへの元の入力を足し算する層
        self.output = BertOutput(config)

    def forward(self, hidden_states, attention_mask, attention_show_flg=False):
        '''
        hidden_states：Embedderモジュールの出力テンソル [batch_size, seq_len, hidden_size]
        attention_mask：Transformerのマスクと同じ働きのマスキング
        attention_show_flg：Self-Attentionの重みを返すかのフラグ
        '''
        if attention_show_flg:
            '''attention_showのときは、attention_probsもリターンする'''
            # Self-Attentionの出力とその重みを計算
            attention_output, attention_probs = self.attention(
                hidden_states, attention_mask, attention_show_flg)
            # 中間層を通す
            intermediate_output = self.intermediate(attention_output)
            # 出力層を通す
            layer_output = self.output(intermediate_output, attention_output)
            return layer_output, attention_probs

        else:
            # Self-Attentionの出力を計算
            attention_output = self.attention(
                hidden_states, attention_mask, attention_show_flg)
            # 中間層を通す
            intermediate_output = self.intermediate(attention_output)
            # 出力層を通す
            layer_output = self.output(intermediate_output, attention_output)
            return layer_output  # [batch_size, seq_length, hidden_size]


In [ ]:
class BertAttention(nn.Module):
    '''BertLayerモジュールのSelf-Attention部分です'''

    def __init__(self, config):
        super(BertAttention, self).__init__()
        # Self-Attention部分の定義
        self.selfattn = BertSelfAttention(config)
        # Self-Attentionの出力を処理する部分の定義
        self.output = BertSelfOutput(config)

    def forward(self, input_tensor, attention_mask, attention_show_flg=False):
        '''
        input_tensor：Embeddingsモジュールもしくは前段のBertLayerからの出力
        attention_mask：Transformerのマスクと同じ働きのマスキングです
        attention_show_flg：Self-Attentionの重みを返すかのフラグ
        '''
        if attention_show_flg:
            '''attention_showのときは、attention_probsもリターンする'''
            # Self-Attentionの出力とその重みを計算
            self_output, attention_probs = self.selfattn(input_tensor, attention_mask, attention_show_flg)
            # Self-Attentionの出力と元の入力を用いて最終出力を計算
            attention_output = self.output(self_output, input_tensor)
            return attention_output, attention_probs

        else:
            # Self-Attentionの出力を計算
            self_output = self.selfattn(input_tensor, attention_mask, attention_show_flg)
            # Self-Attentionの出力と元の入力を用いて最終出力を計算
            attention_output = self.output(self_output, input_tensor)
            return attention_output

In [ ]:
class BertSelfAttention(nn.Module):
    '''BertAttentionのSelf-Attention部分'''

    def __init__(self, config):
        super(BertSelfAttention, self).__init__()

        self.num_attention_heads = config.num_attention_heads
        # num_attention_heads': 12

        self.attention_head_size = int(
            config.hidden_size / config.num_attention_heads)  # 768/12=64
        self.all_head_size = self.num_attention_heads * \
            self.attention_head_size  # = 'hidden_size': 768

        # Self-Attentionの特徴量を作成する全結合層
        self.query = nn.Linear(config.hidden_size, self.all_head_size)
        self.key = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)

        # Dropout
        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

    def transpose_for_scores(self, x):
        '''multi-head Attention用にテンソルの形を変換する
        [batch_size, seq_len, hidden] → [batch_size, 12, seq_len, hidden/12]
        '''
        new_x_shape = x.size()[
            :-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def forward(self, hidden_states, attention_mask, attention_show_flg=False):
        '''
        hidden_states：Embeddingsモジュールもしくは前段のBertLayerからの出力
        attention_mask：Transformerのマスクと同じ働きのマスキングです
        attention_show_flg：Self-Attentionの重みを返すかのフラグ
        '''
        # 入力を全結合層で特徴量変換（注意、multi-head Attentionの全部をまとめて変換しています）
        mixed_query_layer = self.query(hidden_states)
        mixed_key_layer = self.key(hidden_states)
        mixed_value_layer = self.value(hidden_states)

        # multi-head Attention用にテンソルの形を変換
        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        # 特徴量同士を掛け算して似ている度合をAttention_scoresとして求める
        attention_scores = torch.matmul(
            query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / \
            math.sqrt(self.attention_head_size)

        # マスクがある部分にはマスクをかけます
        attention_scores = attention_scores + attention_mask
        # （備考）
        # マスクが掛け算でなく足し算なのが直感的でないですが、このあとSoftmaxで正規化するので、
        # マスクされた部分は-infにしたいです。 attention_maskには、0か-infが
        # もともと入っているので足し算にしています。

        # Attentionを正規化する
        attention_probs = nn.Softmax(dim=-1)(attention_scores)

        # ドロップアウトします
        attention_probs = self.dropout(attention_probs)

        # Attention Mapを掛け算します
        context_layer = torch.matmul(attention_probs, value_layer)

        # multi-head Attentionのテンソルの形をもとに戻す
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[
            :-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)

        # attention_showのときは、attention_probsもリターンする
        if attention_show_flg == True:
            return context_layer, attention_probs
        elif attention_show_flg == False:
            return context_layer

In [ ]:
class BertSelfOutput(nn.Module):
    '''BertSelfAttentionの出力を処理する全結合層'''

    def __init__(self, config):
        super(BertSelfOutput, self).__init__()

        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.LayerNorm = BertLayerNorm(config.hidden_size, eps=1e-12)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        # 'hidden_dropout_prob': 0.1

    def forward(self, hidden_states, input_tensor):
        '''
        hidden_states：BertSelfAttentionの出力テンソル
        input_tensor：Embeddingsモジュールもしくは前段のBertLayerからの出力
        '''
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states

In [ ]:
def gelu(x):
    '''Gaussian Error Linear Unitという活性化関数です。
       LeLUが0でカクっと不連続なので、そこを連続になるように滑らかにした形のLeLUです。
    '''
    return x * 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))


class BertIntermediate(nn.Module):
    ''' BERTのTransformerBlockモジュールのFeedForwardです '''
    def __init__(self, config):
        super(BertIntermediate, self).__init__()

        # 全結合層：'hidden_size': 768、'intermediate_size': 3072
        self.dense = nn.Linear(config.hidden_size, config.intermediate_size)

        # 活性化関数gelu
        self.intermediate_act_fn = gelu

    def forward(self, hidden_states):
        '''
        hidden_states： BertAttentionの出力テンソル
        '''
        hidden_states = self.dense(hidden_states)
        hidden_states = self.intermediate_act_fn(hidden_states)  # GELUによる活性化
        return hidden_states

In [ ]:
class BertOutput(nn.Module):
    '''BERTのTransformerBlockモジュールのFeedForwardです'''

    def __init__(self, config):
        super(BertOutput, self).__init__()

        # 全結合層：'intermediate_size': 3072、'hidden_size': 768
        self.dense = nn.Linear(config.intermediate_size, config.hidden_size)

        self.LayerNorm = BertLayerNorm(config.hidden_size, eps=1e-12)

        # 'hidden_dropout_prob': 0.1
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states, input_tensor):
        '''
        hidden_states： BertIntermediateの出力テンソル
        input_tensor：BertAttentionの出力テンソル
        '''
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states

## BertLayerモジュールの繰り返し部分

BERT_Baseでは、Bert_Layerモジュール(Transformer)を12回繰り返すため、それらをまとめてBertEncoderというクラスにする。

ここでは、単純にBertLayerを12回nn.ModuleListに記載することで、順伝播している。

In [ ]:
# BertLayerモジュールの繰り返し部分モジュールの繰り返し部分です
class BertEncoder(nn.Module):
    def __init__(self, config):
        '''BertLayerモジュールの繰り返し部分モジュールの繰り返し部分です'''
        super(BertEncoder, self).__init__()

        # config.num_hidden_layers の値、すなわち12 個のBertLayerモジュールを作ります
        self.layer = nn.ModuleList([BertLayer(config)
                                    for _ in range(config.num_hidden_layers)])

    def forward(self, hidden_states, attention_mask, output_all_encoded_layers=True, attention_show_flg=False):
        '''
        hidden_states：Embeddingsモジュールの出力
        attention_mask：Transformerのマスクと同じ働きのマスキングです
        output_all_encoded_layers：返り値を全TransformerBlockモジュールの出力にするか、
        それとも、最終層だけにするかのフラグ。
        attention_show_flg：Self-Attentionの重みを返すかのフラグ
        '''

        # 返り値として使うリスト
        all_encoder_layers = []

        # BertLayerモジュールの処理を繰り返す
        for layer_module in self.layer:

            if attention_show_flg == True:
                '''attention_showのときは、attention_probsもリターンする'''
                hidden_states, attention_probs = layer_module(
                    hidden_states, attention_mask, attention_show_flg)
            elif attention_show_flg == False:
                hidden_states = layer_module(
                    hidden_states, attention_mask, attention_show_flg)

            # 返り値にBertLayerから出力された特徴量を12層分、すべて使用する場合の処理
            if output_all_encoded_layers:
                all_encoder_layers.append(hidden_states)

        # 返り値に最後のBertLayerから出力された特徴量だけを使う場合の処理
        if not output_all_encoded_layers:
            all_encoder_layers.append(hidden_states)

        # attention_showのときは、attention_probs（最後の12段目）もリターンする
        if attention_show_flg == True:
            return all_encoder_layers, attention_probs
        elif attention_show_flg == False:
            return all_encoder_layers

## BertPoolerモジュール

BertPoolerは、BertEncoderの出力から、入力文章の１単語目である[CLS]の特徴量テンソル（1*768次元）を取り出し、全結合層を使用して特徴量変換するモジュール。全結合層のあとに活性化関数tanhを利用し、出力を-1~1の範囲にする。出力するテンソルサイズは（batch_size, hidden_size）となる。

In [ ]:
class BertPooler(nn.Module):
    '''入力文章の1単語目[cls]の特徴量を変換して保持するためのモジュール'''

    def __init__(self, config):
        super(BertPooler, self).__init__()

        # 全結合層、'hidden_size': 768
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.activation = nn.Tanh()

    def forward(self, hidden_states):
        # 1単語目の特徴量を取得
        first_token_tensor = hidden_states[:, 0]

        # 全結合層で特徴量変換
        pooled_output = self.dense(first_token_tensor)

        # 活性化関数Tanhを計算
        pooled_output = self.activation(pooled_output)

        return pooled_output

## 動作確認

## BERTモデルの作成

以上を全部つなげて、BERTモデルを作り上げる。

In [ ]:
# 必要なクラスの定義
class BertEmbeddings(nn.Module):
    """単語、位置、トークンタイプの埋め込みを作成します。"""
    # BERTの入力側の埋め込み層。3つを足し合わせる:
    #  - word_embeddings        : 各トークンID→ベクトル
    #  - position_embeddings    : 位置(0,1,2,...)→ベクトル（単語順序情報）
    #  - token_type_embeddings  : セグメントID(文A/文Bなど)→ベクトル
    # さらにLayerNormとDropoutで正規化＆正則化する（Transformer定番の前処理）

    def __init__(self, config):
        super(BertEmbeddings, self).__init__()
        self.word_embeddings = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=config.pad_token_id)  # 語彙IDをhidden_size次元の埋め込みに変換。PADトークンの埋め込みは勾配0扱いになるようpadding_idx指定
        self.position_embeddings = nn.Embedding(config.max_position_embeddings, config.hidden_size)                  # 入力系列内の位置(0〜max_len-1)用の埋め込み
        self.token_type_embeddings = nn.Embedding(config.type_vocab_size, config.hidden_size)                        # セグメント埋め込み（文1=0, 文2=1など）

        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)  # 各トークンベクトルを正規化（BERTはLayerNorm後に残差を足すスタイル）
        self.dropout = nn.Dropout(config.hidden_dropout_prob)                        # 過学習防止用ドロップアウト

    def forward(self, input_ids, token_type_ids):
        seq_length = input_ids.size(1)                                               # 系列長 (batch, seq_len)
        position_ids = torch.arange(seq_length, dtype=torch.long, device=input_ids.device)  # 0..seq_len-1 の位置ID
        position_ids = position_ids.unsqueeze(0).expand_as(input_ids)               # (1,seq_len)→(batch,seq_len) にブロードキャスト

        words_embeddings = self.word_embeddings(input_ids)              # (batch, seq_len, hidden)
        position_embeddings = self.position_embeddings(position_ids)    # (batch, seq_len, hidden)
        token_type_embeddings = self.token_type_embeddings(token_type_ids)  # (batch, seq_len, hidden)

        embeddings = words_embeddings + position_embeddings + token_type_embeddings  # BERTは3つを単純加算
        embeddings = self.LayerNorm(embeddings)                                      # 正規化
        embeddings = self.dropout(embeddings)                                        # ドロップアウト
        return embeddings                                                            # これがEncoderに入る最初の表現

class BertSelfAttention(nn.Module):
    def __init__(self, config):
        super(BertSelfAttention, self).__init__()
        if config.hidden_size % config.num_attention_heads != 0:
            raise ValueError("hidden_sizeはnum_attention_headsで割り切れる必要があります。")

        self.num_attention_heads = config.num_attention_heads                         # ヘッド数
        self.attention_head_size = int(config.hidden_size / config.num_attention_heads)  # 1ヘッドあたりの次元数
        self.all_head_size = self.num_attention_heads * self.attention_head_size         # = hidden_size と同じはず

        self.query = nn.Linear(config.hidden_size, self.all_head_size)  # 入力hiddenからQベクトル群をまとめて出す
        self.key   = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)  #~ Attention確率にかけるdropout（学習時のみ有効）

    def transpose_for_scores(self, x):
        #~ (batch, seq_len, all_head_size) を
        #~ (batch, num_heads, seq_len, head_dim) に並べ替える
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def forward(self, hidden_states, attention_mask, attention_show_flg=False):
        mixed_query_layer = self.query(hidden_states)
        mixed_key_layer   = self.key(hidden_states)
        mixed_value_layer = self.value(hidden_states)

        # ヘッドごとに分割して (B, H, L, Dh) の形に
        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer   = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        # Attentionスコア = QK^T / sqrt(d_k)
        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        attention_scores = attention_scores + attention_mask

        attention_probs = nn.Softmax(dim=-1)(attention_scores)
        attention_probs = self.dropout(attention_probs)

        # Attention重み×V で文脈ベクトルを集約
        context_layer = torch.matmul(attention_probs, value_layer)

        # (B,H,L,Dh) → (B,L,H,Dh) → (B,L,H*Dh=hidden) で元のhidden次元に戻す
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)

        if attention_show_flg:
            return context_layer, attention_probs
        else:
            return context_layer

class BertSelfOutput(nn.Module):
    # Self-Attention の出力に対して線形変換→Dropout→残差接続+LayerNormを行う
    def __init__(self, config):
        super(BertSelfOutput, self).__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)                # ヘッド結合後のベクトルを元のhidden次元に射影
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states

class BertAttention(nn.Module):
    # Self-Attentionブロック全体 = (BertSelfAttention + BertSelfOutput)
    # attention_show_flg=True のときはアテンション重みも返す（可視化用）
    def __init__(self, config):
        super(BertAttention, self).__init__()
        self.self_attention = BertSelfAttention(config)  # マルチヘッドSelf-Attention本体
        self.output = BertSelfOutput(config)            # 残差+LayerNormを含む出力層

    def forward(self, input_tensor, attention_mask, attention_show_flg=False):
        if attention_show_flg:
            self_output, attention_probs = self.self_attention(input_tensor, attention_mask, attention_show_flg)
            attention_output = self.output(self_output, input_tensor)  # 残差+LayerNorm
            return attention_output, attention_probs                   # 文脈表現と各headのAttention確率
        else:
            self_output = self.self_attention(input_tensor, attention_mask)
            attention_output = self.output(self_output, input_tensor)
            return attention_output

class BertIntermediate(nn.Module):
    def __init__(self, config):
        super(BertIntermediate, self).__init__()
        self.dense = nn.Linear(config.hidden_size, config.intermediate_size)
        if isinstance(config.hidden_act, str):
            self.intermediate_act_fn = nn.GELU()
        else:
            self.intermediate_act_fn = config.hidden_act

    def forward(self, hidden_states):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.intermediate_act_fn(hidden_states)
        return hidden_states

class BertOutput(nn.Module):
    # 残差+LayerNorm
    def __init__(self, config):
        super(BertOutput, self).__init__()
        self.dense = nn.Linear(config.intermediate_size, config.hidden_size)
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps) # 残差+LayerNorm
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)            # 次元をhidden_sizeに戻す
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)  # 残差接続
        return hidden_states

class BertLayer(nn.Module):
    # attention_show_flg=True の場合はAttention重みも併せて返す
    def __init__(self, config):
        super(BertLayer, self).__init__()
        self.attention = BertAttention(config)      # Self-Attention + 残差/LN
        self.intermediate = BertIntermediate(config)
        self.output = BertOutput(config)

    def forward(self, hidden_states, attention_mask, attention_show_flg=False):
        if attention_show_flg:
            attention_output, attention_probs = self.attention(hidden_states, attention_mask, attention_show_flg)
            intermediate_output = self.intermediate(attention_output)
            layer_output = self.output(intermediate_output, attention_output)
            return layer_output, attention_probs  # (層出力, この層のattention重み)
        else:
            attention_output = self.attention(hidden_states, attention_mask)
            intermediate_output = self.intermediate(attention_output)
            layer_output = self.output(intermediate_output, attention_output)
            return layer_output

class BertEncoder(nn.Module):
    def __init__(self, config):
        super(BertEncoder, self).__init__()
        self.layer = nn.ModuleList([BertLayer(config)
                                    for _ in range(config.num_hidden_layers)])  # Transformer層をリスト化

    def forward(self, hidden_states, attention_mask, output_all_encoded_layers=True, attention_show_flg=False):
        all_encoder_layers = []
        all_attention_probs = []

        for layer_module in self.layer:
            if attention_show_flg:
                hidden_states, attention_probs = layer_module(
                    hidden_states, attention_mask, attention_show_flg)
                all_attention_probs.append(attention_probs)  # 各層のattention確率を保存
            else:
                hidden_states = layer_module(hidden_states, attention_mask)

            if output_all_encoded_layers:
                all_encoder_layers.append(hidden_states)     # 各層の出力を順に保存

        if not output_all_encoded_layers:
            all_encoder_layers.append(hidden_states)         # 最終層だけ返したい場合は末尾だけ

        if attention_show_flg:
            return all_encoder_layers, all_attention_probs   # 出力群とAttention群両方
        else:
            return all_encoder_layers                        # 出力群のみ

class BertPooler(nn.Module):
    # プール処理: 文全体を代表する1つのベクトルを作る
    # BERTの[CLS]トークン（先頭トークン）の隠れ状態を取り出し、線形+Tanhで変換
    def __init__(self, config):
        super(BertPooler, self).__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)  # [CLS]ベクトルを同次元で変換
        self.activation = nn.Tanh()                                     # 活性化はtanh固定(BERT標準)

    def forward(self, hidden_states):
        # 最初のトークンの隠れ状態を用いてプーリングします
        first_token_tensor = hidden_states[:, 0]     # [CLS]トークン位置のベクトル (batch, hidden)
        pooled_output = self.dense(first_token_tensor)
        pooled_output = self.activation(pooled_output)
        return pooled_output                         # (batch, hidden)

class BertModel(nn.Module):
    '''モジュールを全部つなげたBERTモデル'''
    # Embedding → Encoder(多層Transformer) → Pooler([CLS]から文ベクトル)

    def __init__(self, config):
        super(BertModel, self).__init__()

        # 3つのモジュールを作成
        self.embeddings = BertEmbeddings(config)  # 単語/位置/セグメント埋め込み + LayerNorm + Dropout
        self.encoder = BertEncoder(config)        # Transformerエンコーダ本体
        self.pooler = BertPooler(config)          # [CLS]から文全体の表現を作る

    def forward(self, input_ids, token_type_ids=None, attention_mask=None, output_all_encoded_layers=True, attention_show_flg=False):
        '''
        input_ids： [batch_size, sequence_length]の文章の単語IDの羅列
        token_type_ids： [batch_size, sequence_length]の、各単語が1文目なのか、2文目なのかを示すid
        attention_mask：Transformerのマスクと同じ働きのマスキングです
        output_all_encoded_layers：最終出力に12段のTransformerの全部をリストで返すか、最後だけかを指定
        attention_show_flg：Self-Attentionの重みを返すかのフラグ
        '''

        # Attentionのマスクと文の1文目、2文目のidが無ければ作成する
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)    # Noneなら全部1（=全トークン有効）
        if token_type_ids is None:
            token_type_ids = torch.zeros_like(input_ids)   # Noneなら全部0（=全部1文目扱い）

        # マスクの変形　[minibatch, 1, 1, seq_length]にする
        # Self-Attentionの (B,H,L,L) スコアにブロードキャストできるように4次元に拡張
        extended_attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)
        extended_attention_mask = extended_attention_mask.to(
            dtype=torch.float32)
        extended_attention_mask = (1.0 - extended_attention_mask) * -10000.0

        # 順伝搬させる
        embedding_output = self.embeddings(input_ids, token_type_ids)

        if attention_show_flg == True:
            encoded_layers, attention_probs = self.encoder(
                embedding_output, extended_attention_mask, output_all_encoded_layers, attention_show_flg)
        # attention_probs は各層ごとの (B, num_heads, L, L)
        else:
            encoded_layers = self.encoder(
                embedding_output, extended_attention_mask, output_all_encoded_layers)
            # encoded_layers はリスト: 各層出力 or 最終層

        pooled_output = self.pooler(encoded_layers[-1])  # 最終層の[CLS]から文ベクトルを作る

        if not output_all_encoded_layers:
            encoded_layers = encoded_layers[-1]          # Trueでなければ最終層だけ返す形に整える

        if attention_show_flg == True:
            return encoded_layers, pooled_output, attention_probs[-1]  # 最終層のAttention重みだけ返す仕様
        else:
            return encoded_layers, pooled_output

# 動作確認
# 入力の用意
input_ids = torch.LongTensor([[31, 51, 12, 23, 99],
                              [15, 5, 1, 0, 0]])
attention_mask = torch.LongTensor([[1, 1, 1, 1, 1],
                                   [1, 1, 1, 0, 0]])
token_type_ids = torch.LongTensor([[0, 0, 1, 1, 1],
                                   [0, 1, 1, 1, 1]])

# BERTの設定を用意
config = BertConfig()

# BERTモデルを作る
net = BertModel(config)  # 上で定義したBertModelをインスタンス化

# 順伝搬させる
encoded_layers, pooled_output, attention_probs = net(
    input_ids, token_type_ids, attention_mask, output_all_encoded_layers=False, attention_show_flg=True)

print("encoded_layersのテンソルサイズ：", encoded_layers.shape)      # 各トークンごとの最終隠れ状態
print("pooled_outputのテンソルサイズ：", pooled_output.shape)        # [CLS]ベースの文章ベクトル
print("attention_probsのテンソルサイズ：", attention_probs.shape)    # Self-Attentionの重み行列


encoded_layersのテンソルサイズ： torch.Size([2, 5, 768])
pooled_outputのテンソルサイズ： torch.Size([2, 768])
attention_probsのテンソルサイズ： torch.Size([2, 12, 5, 5])


#3. BERTを用いたベクトル表現の比較

## 学習済みモデルのロード

In [ ]:
# 学習済みモデルのロード
weights_path = "./weights/pytorch_model.bin"
loaded_state_dict = torch.load(weights_path)

for s in loaded_state_dict.keys():
    print(s)

bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.gamma
bert.embeddings.LayerNorm.beta
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.gamma
bert.encoder.layer.0.attention.output.LayerNorm.beta
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.gamma
bert.encoder.layer.0.output.LayerNorm.beta
bert.encoder.layer.1.attention.self.query.weight
bert.encode

In [ ]:
# モデルの用意
config = BertConfig()
net = BertModel(config)
net.eval()

# 現在のネットワークモデルのパラメータ名
param_names = []  # パラメータの名前を格納していく

for name, param in net.named_parameters():
    param_names.append(name)

In [ ]:
# state_dictの名前が違うので前から順番に代入する
# 現在のネットワークの情報をコピーして新たなstate_dictを作成
new_state_dict = net.state_dict().copy()

# 新たなstate_dictに学習済みの値を代入
for index, (key_name, value) in enumerate(loaded_state_dict.items()):
    name = param_names[index]  # 現在のネットワークでのパラメータ名を取得
    new_state_dict[name] = value  # 値を入れる
    print(str(key_name)+" → "+str(name))  # 何から何に入ったかを表示

    # 現在のネットワークのパラメータを全部ロードしたら終える
    if index+1 >= len(param_names):
        break

# 新たなstate_dictを実装したBERTモデルに与える
net.load_state_dict(new_state_dict)

bert.embeddings.word_embeddings.weight → embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight → embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight → embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.gamma → embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.beta → embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight → encoder.layer.0.attention.self_attention.query.weight
bert.encoder.layer.0.attention.self.query.bias → encoder.layer.0.attention.self_attention.query.bias
bert.encoder.layer.0.attention.self.key.weight → encoder.layer.0.attention.self_attention.key.weight
bert.encoder.layer.0.attention.self.key.bias → encoder.layer.0.attention.self_attention.key.bias
bert.encoder.layer.0.attention.self.value.weight → encoder.layer.0.attention.self_attention.value.weight
bert.encoder.layer.0.attention.self.value.bias → encoder.layer.0.attention.self_attention.value.bias
bert.encoder.layer.0.att

<All keys matched successfully>

## BERT用のTokenizerの実装

BERTでは、Toransformerのように単純にスペースで分割するのではなく、サブワードの概念で単語を分割している。

In [ ]:
# vocabファイルを読み込み、
def load_vocab(vocab_file):
    """text形式のvocabファイルの内容を辞書に格納します"""
    vocab = collections.OrderedDict()  # (単語, id)の順番の辞書変数
    ids_to_tokens = collections.OrderedDict()  # (id, 単語)の順番の辞書変数
    index = 0

    with open(vocab_file, "r", encoding="utf-8") as reader:
        while True:
            token = reader.readline()
            if not token:
                break  # ファイル末尾まで読んだら終了
            token = token.strip()  # 改行などを除去

            # 格納
            vocab[token] = index           # 単語→ID
            ids_to_tokens[index] = token   # ID→単語
            index += 1

    return vocab, ids_to_tokens
    # これ以降のTokenizerでこの対応表を使ってID化 / 復元する

BERT用の単語分割クラスを実装

In [ ]:
# Tokenizerの定義
class BasicTokenizer(object):
    """基本的なTokenizer。空白で単語を分割し、必要に応じて小文字化を行います。"""

    def __init__(self, do_lower_case=True, never_split=None):
        self.do_lower_case = do_lower_case  # Trueなら英字を小文字化する
        if never_split is None:
            self.never_split = set()
        else:
            self.never_split = set(never_split)

    def tokenize(self, text):
        # 空白で分割
        tokens = text.strip().split()
        split_tokens = []
        for token in tokens:
            if self.do_lower_case and token not in self.never_split:
                # do_lower_case=True かつ 特殊トークンでない場合は小文字化
                token = token.lower()
            split_tokens.append(token)
        return split_tokens
        # 出力は単語のリスト

class WordpieceTokenizer(object):
    """WordPieceトークナイザー。未知の単語を分割します。"""

    def __init__(self, vocab, unk_token="[UNK]", max_input_chars_per_word=100):
        self.vocab = vocab                                # load_vocabで作った単語→IDの辞書
        self.unk_token = unk_token                        # 分割不可能な場合に使うトークン
        self.max_input_chars_per_word = max_input_chars_per_word  # あまりに長い文字列は強制的に[UNK]

    def tokenize(self, text):
        output_tokens = []
        for token in text:
            chars = list(token)
            if len(chars) > self.max_input_chars_per_word:
                output_tokens.append(self.unk_token)
                continue

            is_bad = False       # 分割に失敗したかどうか
            start = 0
            sub_tokens = []
            while start < len(chars):
                end = len(chars)
                cur_substr = None
                while start < end:
                    # 文字のスライスをとって辞書にあるか確認する
                    # WordPieceでは先頭以外は "##" を付けてサブワード扱いにする
                    substr = "".join(chars[start:end])
                    if start > 0:
                        substr = "##" + substr
                    if substr in self.vocab:
                        cur_substr = substr
                        break
                    end -= 1     # マッチしないなら1文字短くして再トライ
                if cur_substr is None:
                    is_bad = True
                    break
                sub_tokens.append(cur_substr)
                start = end      # マッチした部分の終わりから次を再開
            if is_bad:
                output_tokens.append(self.unk_token)
            else:
                output_tokens.extend(sub_tokens)
        return output_tokens

class BertTokenizer(object):
    '''BERT用の文章の単語分割クラスを実装'''

    def __init__(self, vocab_file, do_lower_case=True):
        '''
        vocab_file：ボキャブラリーへのパス
        do_lower_case：前処理で単語を小文字化するかどうか
        '''

        # ボキャブラリーのロード
        self.vocab, self.ids_to_tokens = load_vocab(vocab_file)

        # 分割処理の関数を定義
        never_split = ("[UNK]", "[SEP]", "[PAD]", "[CLS]", "[MASK]")
        # (注釈)上記の単語は途中で分割させない。これで一つの単語とみなす

        self.basic_tokenizer = BasicTokenizer(do_lower_case=do_lower_case,
                                              never_split=never_split)
        self.wordpiece_tokenizer = WordpieceTokenizer(vocab=self.vocab)

    def tokenize(self, text):
        '''文章を単語に分割する関数'''
        split_tokens = []  # 分割後の単語たち
        for token in self.basic_tokenizer.tokenize(text):
            sub_tokens = self.wordpiece_tokenizer.tokenize(token)
            split_tokens.extend(sub_tokens)
        return split_tokens

    def convert_tokens_to_ids(self, tokens):
        """分割された単語リストをIDに変換する関数"""
        ids = []
        for token in tokens:
            ids.append(self.vocab.get(token, self.vocab.get("[UNK]")))
        return ids

    def convert_ids_to_tokens(self, ids):
        """IDを単語に変換する関数"""
        tokens = []
        for i in ids:
            tokens.append(self.ids_to_tokens.get(i, "[UNK]"))
        return tokens

## bankの文脈による意味変化を単語ベクトルとして求める。

In [ ]:
from transformers import BertTokenizer

# トークナイザーの作成
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# 文章の定義
text_1 = "I accessed the bank account."
text_2 = "He transferred the deposit money into the bank account."
text_3 = "We play soccer at the bank of the river."

# 文章をトークン化
tokenized_text_1 = tokenizer.tokenize(text_1)
tokenized_text_2 = tokenizer.tokenize(text_2)
tokenized_text_3 = tokenizer.tokenize(text_3)

# 確認
print("tokenized_text_1:", tokenized_text_1)
print("tokenized_text_2:", tokenized_text_2)
print("tokenized_text_3:", tokenized_text_3)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenized_text_1: ['i', 'accessed', 'the', 'bank', 'account', '.']
tokenized_text_2: ['he', 'transferred', 'the', 'deposit', 'money', 'into', 'the', 'bank', 'account', '.']
tokenized_text_3: ['we', 'play', 'soccer', 'at', 'the', 'bank', 'of', 'the', 'river', '.']


In [ ]:
# 各文章のbankの位置
bank_posi_1 = np.where(np.array(tokenized_text_1) == "bank")[0][0]
bank_posi_2 = np.where(np.array(tokenized_text_2) == "bank")[0][0]
bank_posi_3 = np.where(np.array(tokenized_text_3) == "bank")[0][0]

# 単語をIDに変換する
indexed_tokens_1 = tokenizer.convert_tokens_to_ids(tokenized_text_1)
indexed_tokens_2 = tokenizer.convert_tokens_to_ids(tokenized_text_2)
indexed_tokens_3 = tokenizer.convert_tokens_to_ids(tokenized_text_3)

# リストをPyTorchのテンソルに
tokens_tensor_1 = torch.tensor([indexed_tokens_1])
tokens_tensor_2 = torch.tensor([indexed_tokens_2])
tokens_tensor_3 = torch.tensor([indexed_tokens_3])

# bankの単語IDの取得
bank_word_id = tokenizer.convert_tokens_to_ids(["bank"])[0]

# 確認
print("tokens_tensor_1:", tokens_tensor_1)
print("tokens_tensor_2:", tokens_tensor_2)
print("tokens_tensor_3:", tokens_tensor_3)

tokens_tensor_1: tensor([[ 1045, 11570,  1996,  2924,  4070,  1012]])
tokens_tensor_2: tensor([[ 2002,  4015,  1996, 12816,  2769,  2046,  1996,  2924,  4070,  1012]])
tokens_tensor_3: tensor([[2057, 2377, 4715, 2012, 1996, 2924, 1997, 1996, 2314, 1012]])


In [ ]:
# 文章をBERTで処理
with torch.no_grad():
    encoded_layers_1, pooled_output_1 = net(tokens_tensor_1, output_all_encoded_layers=True)
    encoded_layers_2, pooled_output_2 = net(tokens_tensor_2, output_all_encoded_layers=True)
    encoded_layers_3, pooled_output_3 = net(tokens_tensor_3, output_all_encoded_layers=True)

# 各層の出力（hidden states）を取得
print(f"encoded_layers_1の長さ: {len(encoded_layers_1)}")
print(f"encoded_layers_2の長さ: {len(encoded_layers_2)}")
print(f"encoded_layers_3の長さ: {len(encoded_layers_3)}")

encoded_layers_1の長さ: 12
encoded_layers_2の長さ: 12
encoded_layers_3の長さ: 12


In [ ]:
# bankの初期の単語ベクトル表現
# これはEmbeddingsモジュールから取り出し、単語bankのidに応じた単語ベクトルなので3文で共通している
bank_vector_0 = net.embeddings.word_embeddings.weight[bank_word_id]

# 文章1のBertLayerモジュール1段目から出力されるbankの特徴量ベクトル
bank_vector_1_1 = encoded_layers_1[0][0, bank_posi_1]

# 文章1のBertLayerモジュール最終12段目から出力されるのbankの特徴量ベクトル
bank_vector_1_12 = encoded_layers_1[11][0, bank_posi_1]

# 文章2、3も同様に
bank_vector_2_1 = encoded_layers_2[0][0, bank_posi_2]
bank_vector_2_12 = encoded_layers_2[11][0, bank_posi_2]
bank_vector_3_1 = encoded_layers_3[0][0, bank_posi_3]
bank_vector_3_12 = encoded_layers_3[11][0, bank_posi_3]

In [ ]:
# コサイン類似度を計算

print("bankの初期ベクトル と 文章1の1段目のbankの類似度：",
      F.cosine_similarity(bank_vector_0, bank_vector_1_1, dim=0))
print("bankの初期ベクトル と 文章1の12段目のbankの類似度：",
      F.cosine_similarity(bank_vector_0, bank_vector_1_12, dim=0))

print("文章1の1層目のbank と 文章2の1段目のbankの類似度：",
      F.cosine_similarity(bank_vector_1_1, bank_vector_2_1, dim=0))
print("文章1の1層目のbank と 文章3の1段目のbankの類似度：",
      F.cosine_similarity(bank_vector_1_1, bank_vector_3_1, dim=0))

print("文章1の12層目のbank と 文章2の12段目のbankの類似度：",
      F.cosine_similarity(bank_vector_1_12, bank_vector_2_12, dim=0))
print("文章1の12層目のbank と 文章3の12段目のbankの類似度：",
      F.cosine_similarity(bank_vector_1_12, bank_vector_3_12, dim=0))

bankの初期ベクトル と 文章1の1段目のbankの類似度： tensor(0.6344, grad_fn=<SumBackward1>)
bankの初期ベクトル と 文章1の12段目のbankの類似度： tensor(0.0416, grad_fn=<SumBackward1>)
文章1の1層目のbank と 文章2の1段目のbankの類似度： tensor(0.9018)
文章1の1層目のbank と 文章3の1段目のbankの類似度： tensor(0.7229)
文章1の12層目のbank と 文章2の12段目のbankの類似度： tensor(0.8053)
文章1の12層目のbank と 文章3の12段目のbankの類似度： tensor(0.5785)


#4. BERTの学習・推論、判定根拠の可視化を実装

## IMDbデータを読み込み、DataLoaderを作成（BERTのTokenizerを使用）

In [ ]:
# 前処理関数の定義
def preprocess_text(text):
    '''IMDbの前処理'''
    # 改行コードを消去
    text = re.sub('<br />', '', text)

    # カンマ、ピリオド以外の記号をスペースに置換
    for p in string.punctuation:
        if p not in [".", ","]:
            text = text.replace(p, " ")

    # ピリオドなどの前後にはスペースを入れておく
    text = text.replace(".", " . ")
    text = text.replace(",", " , ")
    return text

# トークナイザーの作成
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# カスタムデータセットクラスの定義
class CustomDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = [preprocess_text(text) for text in texts]  # 前処理を適用
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        # トークン化とエンコード
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
            return_attention_mask=True
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# サンプルのテキストとラベルを定義
texts = [
    "This is the first text.",
    "Here is another example of text.",
    "This text is the third sample."
]
labels = [0, 1, 0]  # サンプルのラベル（0または1）

# データセットの作成
dataset = CustomDataset(texts, labels, tokenizer, max_length=32)

# データローダーの作成
dataloader = DataLoader(dataset, batch_size=2, shuffle=False)

# ミニバッチの取得
batch = next(iter(dataloader))

In [ ]:
# ミニバッチの1文目を確認
text_minibatch_1 = batch['input_ids'][0].tolist()

# IDを単語に戻す
text = tokenizer.convert_ids_to_tokens(text_minibatch_1)

print(text)

['[CLS]', 'this', 'is', 'the', 'first', 'text', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


## BERTのファインチューニングに向けた設定

In [ ]:
# IMDbデータの読み込み
train_data = pd.read_csv('./data/IMDb_train.tsv', sep='\t', header=None)
test_data = pd.read_csv('./data/IMDb_test.tsv', sep='\t', header=None)

# テキストとラベルをそれぞれリストに変換
train_texts, train_labels = train_data[0].tolist(), train_data[1].astype(int).tolist()
test_texts, test_labels = test_data[0].tolist(), test_data[1].astype(int).tolist()

# 訓練データを訓練用と検証用に分割（80%訓練、20%検証）
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.2, random_state=1234
)

# パラメータ設定
max_length = 256
batch_size = 32

# データセットの作成
train_dataset = CustomDataset(train_texts, train_labels, tokenizer, max_length)
val_dataset = CustomDataset(val_texts, val_labels, tokenizer, max_length)
test_dataset = CustomDataset(test_texts, test_labels, tokenizer, max_length)

# データローダーの作成
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# データローダーの辞書を作成
dataloaders_dict = {
    'train': train_loader,
    'val': val_loader
}

# データローダーのバッチを確認（訓練データ）
for batch in train_loader:
    print(batch)
    break  # 最初のバッチのみ表示

# ミニバッチの1文目を確認
text_minibatch_1 = batch['input_ids'][0].tolist()  # 1文目のinput_idsを取得

# IDを単語に戻す
text = tokenizer.convert_ids_to_tokens(text_minibatch_1)
print(text)

# BERTモデルの読み込み
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.train()  # 訓練モードに設定

# 1. すべてのパラメータの勾配計算を有効化
for param in model.parameters():
    param.requires_grad = True

# 2. 凍結したいパラメータの勾配計算を無効化
# 最後の層と分類器のみを学習対象とする
for name, param in model.named_parameters():
    if not any(n in name for n in ['bert.encoder.layer.11', 'classifier']):
        param.requires_grad = False

# 最適化手法の設定
optimizer = optim.Adam([
    {'params': model.bert.encoder.layer[-1].parameters(), 'lr': 5e-5},
    {'params': model.classifier.parameters(), 'lr': 5e-5}
], betas=(0.9, 0.999))

# 損失関数の設定
criterion = nn.CrossEntropyLoss()

{'input_ids': tensor([[  101,  1996,  2087,  ...,     0,     0,     0],
        [  101,  2076,  4537,  ...,  2023,  2828,   102],
        [  101,  2228,  9267,  ...,     0,     0,     0],
        ...,
        [  101,  1045,  2031,  ...,  2075,  1040,   102],
        [  101,  2023,  3118,  ...,  2839,  2061,   102],
        [  101,  1045,  2001,  ..., 12722,  2100,   102]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]]), 'label': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1])}
['[CLS]', 'the', 'most', 'agile', 'fat', 'guy', 'in', 'martial', 'arts', 'does', 'it', 'again', '.', 'an', 'early', 'sam', '##mo', 'film', 'that', 'has', 'him', 'im', '##itating', 'his', 'character', 's', 'hero', ',', 'bruce', 'lee', ',', 'sam', '##mo', 'is', 'am

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 学習・検証を実施

In [ ]:
# 学習関数の定義
def train_model(net, dataloaders_dict, criterion, optimizer, num_epochs):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print("使用デバイス：", device)
    print('-----start-------')
    net.to(device)
    # モデル本体を device (GPU/CPU) 側に転送する

    torch.backends.cudnn.benchmark = True
    # 入力サイズがほぼ一定なら、CuDNNの最適化を有効化して畳み込みなどを高速化

    for epoch in range(num_epochs):
      # エポックループ（num_epochs 回まわす）
        print(f'Epoch {epoch+1}/{num_epochs}')

        for phase in ['train', 'val']:
            # phase='train' のときは学習、'val' のときは検証
            if phase == 'train':
                net.train()
                # trainモード: Dropout有効 / BatchNormが学習用の挙動になる
            else:
                net.eval()
                # evalモード: 勾配は計算しない想定 / Dropout停止 / BNは推論モード
            epoch_loss = 0.0
            epoch_corrects = 0
            iteration = 1
            t_iter_start = time.time()
            # イテレーション時間計測用に現在時刻を保存

            for batch in dataloaders_dict[phase]:
            # Hugging Face系DataLoader想定: batch は dict で 'input_ids', 'attention_mask', 'label' を持つ

                # デバッグ情報を出力
                print(f"Phase: {phase}, Batch keys: {batch.keys()}")
                print(f"input_ids dtype: {batch['input_ids'].dtype}")
                print(f"input_ids shape: {batch['input_ids'].shape}")
                print(f"label dtype: {batch['label'].dtype}")
                print(f"attention_mask dtype: {batch['attention_mask'].dtype}")

                inputs = batch['input_ids'].to(device)
                labels = batch['label'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                # バッチ中の各テンソルをGPU/CPUに移動する

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                   # train時だけ勾配を追跡し、val時は勾配を追跡しない
                    outputs = net(input_ids=inputs, attention_mask=attention_mask, labels=labels)
                    loss = outputs.loss
                    # CrossEntropy系の損失 (labels を渡した場合に自動で計算される想定)
                    logits = outputs.logits
                    # 分類用の生の出力 (batch_size, num_labels)
                    _, preds = torch.max(logits, 1)
                    # 各サンプルごとにスコアが最大のクラスIDを予測ラベルとして取り出す

                    if phase == 'train':
                        loss.backward()
                        # 逆伝播（損失から勾配を計算）
                        optimizer.step()
                        # optimizer更新でモデルのパラメータを1ステップ更新
                        if (iteration % 10 == 0):
                            # 10イテレーションごとに進捗ログを出す
                            t_iter_finish = time.time()
                            duration = t_iter_finish - t_iter_start  # 直近10iterにかかった秒数
                            acc = (torch.sum(preds == labels.data)).double() / len(labels)
                            # このミニバッチ単位での正解率
                            print(f'イテレーション {iteration} || Loss: {loss.item():.4f} || 10iter: {duration:.4f} sec. || 正解率: {acc:.4f}')
                            t_iter_start = time.time()
                    iteration += 1

                    # epoch全体の集計用に損失と正解数を加算
                    epoch_loss += loss.item() * len(inputs)
                    epoch_corrects += torch.sum(preds == labels.data)

                # 最初のバッチ処理後にループを抜ける（デバッグ用）
                # break  # デバッグが終わったらコメントアウトまたは削除

            epoch_loss = epoch_loss / len(dataloaders_dict[phase].dataset)
            epoch_acc = epoch_corrects.double() / len(dataloaders_dict[phase].dataset)
            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            # phaseごとの平均損失と正解率を表示
            # train: 学習の進み具合、val: 汎化性能のチェック

        # 最初のエポック後にループを抜ける（デバッグ用）
        # break  # デバッグが終わったらコメントアウトまたは削除

    return net

In [ ]:
# 学習・検証を実行する
num_epochs = 2
net_trained = train_model(model, dataloaders_dict, criterion, optimizer, num_epochs=num_epochs)

使用デバイス： cuda:0
-----start-------
Epoch 1/2
Phase: train, Batch keys: dict_keys(['input_ids', 'attention_mask', 'label'])
input_ids dtype: torch.int64
input_ids shape: torch.Size([32, 256])
label dtype: torch.int64
attention_mask dtype: torch.int64
Phase: train, Batch keys: dict_keys(['input_ids', 'attention_mask', 'label'])
input_ids dtype: torch.int64
input_ids shape: torch.Size([32, 256])
label dtype: torch.int64
attention_mask dtype: torch.int64
Phase: train, Batch keys: dict_keys(['input_ids', 'attention_mask', 'label'])
input_ids dtype: torch.int64
input_ids shape: torch.Size([32, 256])
label dtype: torch.int64
attention_mask dtype: torch.int64
Phase: train, Batch keys: dict_keys(['input_ids', 'attention_mask', 'label'])
input_ids dtype: torch.int64
input_ids shape: torch.Size([32, 256])
label dtype: torch.int64
attention_mask dtype: torch.int64
Phase: train, Batch keys: dict_keys(['input_ids', 'attention_mask', 'label'])
input_ids dtype: torch.int64
input_ids shape: torch.Size([3

In [ ]:
net_trained.save_pretrained('./weights/bert_fine_tuning_IMDb')

## Attentionの可視化

In [ ]:
# デバイスの設定
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# モデルのロードと設定
model = BertForSequenceClassification.from_pretrained(
    './weights/bert_fine_tuning_IMDb',  # 修正
    output_attentions=True
)
model.to(device)
model.eval()

# トークナイザーの設定
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# データローダーの設定（バッチサイズを変更可能に）
def create_test_loader(batch_size):
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    return test_loader

# テストデータでの正解率を計算する関数
def evaluate_model(model, test_loader):
    model.eval()
    epoch_corrects = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(test_loader):
            inputs = batch['input_ids'].to(device)
            labels = batch['label'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids=inputs, attention_mask=attention_mask)
            logits = outputs.logits
            _, preds = torch.max(logits, 1)

            epoch_corrects += torch.sum(preds == labels).item()
            total += labels.size(0)

    epoch_acc = epoch_corrects / total
    print('テストデータ{}個での正解率：{:.4f}'.format(total, epoch_acc))
    return epoch_acc

# バッチサイズ32で評価
batch_size = 32
test_loader = create_test_loader(batch_size)
evaluate_model(model, test_loader)

# テストデータから1つのバッチを取得
batch = next(iter(test_loader))

# インデックスを指定してデータを取得
index = 0  # 可視化したいデータのインデックス
inputs = batch['input_ids'][index].unsqueeze(0).to(device)
labels = batch['label'][index].unsqueeze(0).to(device)
attention_mask = batch['attention_mask'][index].unsqueeze(0).to(device)

# モデルの出力を取得（Attentionも取得）
outputs = model(input_ids=inputs, attention_mask=attention_mask)
logits = outputs.logits
attentions = outputs.attentions  # 各層のAttentionが格納されている

# 予測ラベルの取得
_, preds = torch.max(logits, 1)

# Attentionを可視化する関数
def visualize_attention(input_ids, attentions, tokenizer):
    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze().tolist())
    # 各層の[CLS]トークンへのAttentionを取得
    cls_attentions = [attn[0, :, 0, :].cpu().detach().numpy() for attn in attentions]
    html = ''
    for layer, attn in enumerate(cls_attentions):
        # 各ヘッドのAttentionの平均を計算
        attn_mean = attn.mean(axis=0)
        attn_mean /= attn_mean.max()
        html += f'Layer {layer+1} Attention<br>'
        for token, score in zip(tokens, attn_mean):
            color = int(255 * (1 - score))
            html += f'<span style="background-color: rgb(255, {color}, {color})">{token} </span>'
        html += '<br><br>'
    return html

# Attentionの可視化
html_output = visualize_attention(inputs, attentions, tokenizer)
HTML(html_output)

 98%|█████████▊| 384/391 [03:57<00:04,  1.55it/s]